In [95]:
from rich.console import Console
from rich.syntax import Syntax
from IPython.display import Markdown

import pandas as pd
import pickle

In [97]:
df_1 = pd.read_csv('vulnerable_verified_smart_contracts_dataset_with_line_numbers_per_vulnerability.csv')
df_2 = pd.read_csv('vulnerable_verified_smart_contracts_dataset_with_line_numbers.csv')

df_1['source_code'] = None
for index, row in df_2.iterrows():
    file_path = row['file_path']
    contract_address = row['contract_address']
    mask = (df_1['file_path'] == file_path) & (df_1['contract_address'] == contract_address)
    df_1.loc[mask, 'source_code'] = row['source_code']

df = df_1
df.to_csv('vulnerable_verified_smart_contracts_dataset_with_line_numbers_per_vulnerability_with_source_code.csv', index=False)

In [98]:
curr_df = pd.read_csv('vulnerable_verified_smart_contracts_dataset_with_line_numbers_per_vulnerability_with_source_code.csv')

In [107]:
def check_code_for_correctness(solidity_code, highlight_lines):
    console = Console()
    
    # Create a Syntax object with highlighting
    syntax = Syntax(solidity_code, "solidity", line_numbers=True, highlight_lines=highlight_lines)
    
    # Print formatted code with highlights
    console.print(syntax)


indices_list = []
def get_indices_list(curr_df):
    global indices_list
    indices_list = curr_df.index.tolist()

def get_claude_output(pkl_file_path, file_path, contract_address):
    print(pkl_file_path)
    with open(pkl_file_path, 'rb') as file:
        print('key')
        data = pickle.load(file)
        for item in data:
            if item['file_path'] == file_path and item['contract_address'] == contract_address:
                return item['response'][-1].text

def check_one_by_one(curr_df):
    curr_index = indices_list.pop()
    file_path = curr_df.loc[curr_index, 'file_path']
    contract_address = curr_df.loc[curr_index, 'contract_address']
    print("index: ", curr_index)
    print("file_path: ", file_path)
    print("contract_address: ", contract_address)
    
    soldetector = curr_df.loc[curr_index, 'soldetector']
    slither = curr_df.loc[curr_index, 'slither']
    oyente = curr_df.loc[curr_index, 'oyente']
    smartcheck = curr_df.loc[curr_index, 'smartcheck']

    need_intervention = curr_df.loc[curr_index, 'need_intervention']
    if need_intervention:
        claude = curr_df.loc[curr_index, 'claude']
        claude_complete = curr_df.loc[curr_index, 'claude_complete']

    final_line_numbers = curr_df.loc[curr_index, 'final_line_numbers']

    highlight = set()

    print("\nvulnerability: ", curr_df.loc[curr_index, 'vuln'], '\n')

    if isinstance(soldetector, str):
        print("soldetector: ", soldetector)
        soldetector = eval(soldetector)
        highlight = highlight.union(soldetector)

    if isinstance(slither, str):
        print("slither: ", slither)
        slither = eval(slither)
        highlight = highlight.union(slither)

    if isinstance(oyente, str):
        print("oyente: ", oyente)
        oyente = eval(oyente)
        highlight = highlight.union(oyente)

    if isinstance(smartcheck, str):
        print("smartcheck: ", smartcheck)
        smartcheck = eval(smartcheck)
        highlight = highlight.union(smartcheck)

    if need_intervention:
        if isinstance(claude, str):
            print("\nclaude: ", claude)
            claude = eval(claude)
            highlight = highlight.union(claude)
            print("claude_complete: ", claude_complete)

    print("\nfinal_line_numbers: ", final_line_numbers)

    source_code = curr_df.loc[curr_index, 'source_code']
    check_code_for_correctness(source_code, highlight)

    if need_intervention:
        curr_df_name = curr_df.loc[curr_index, 'type']
        if isinstance(claude, set):
            print("\nclaude output: \n")
            claude_output = get_claude_output(f'claude_outputs/{curr_df.loc[curr_index, 'pkl_file_path']}', file_path, contract_address)
            display(Markdown(claude_output))
    else:
        print("claude column is not a string", file_path, contract_address)


In [109]:
get_indices_list(curr_df)

In [111]:
check_one_by_one(curr_df)

index:  821
file_path:  MultiSend.sol
contract_address:  0x3d7c33c71996540755f4ae6d7454dea0344f89c0

vulnerability:  NC 

soldetector:  {35}
slither:  {36}
smartcheck:  {35}

claude:  {36, 35}
claude_complete:  True

final_line_numbers:  {36, 35}


   1 pragma solidity ^0.4.18;                                                                                      
   2                                                                                                               
   3 /**                                                                                                           
   4  * @title ERC20Basic                                                                                          
   5  * @dev Simpler version of ERC20 interface                                                                    
   6  * @dev see https://github.com/ethereum/EIPs/issues/179                                                       
   7  */                                                                                                           
   8 contract ERC20Basic {                                                                                         
   9   function totalSupply() public view returns (uint256);                                                       
  10   function balanceOf(address who) public view returns (uint256);                                              
  11   function transfer(address to, uint256 value) public returns (bool);                                         
  12   event Transfer(address indexed from, address indexed to, uint256 value);                                    
  13 }                                                                                                             
  14                                                                                                               
  15                                                                                                               
  16 /**                                                                                                           
  17  * @title ERC20 interface                                                                                     
  18  * @dev see https://github.com/ethereum/EIPs/issues/20                                                        
  19  */                                                                                                           
  20 contract ERC20 is ERC20Basic {                                                                                
  21   function allowance(address owner, address spender) public view returns (uint256);                           
  22   function transferFrom(address from, address to, uint256 value) public returns (bool);                       
  23   function approve(address spender, uint256 value) public returns (bool);                                     
  24   event Approval(address indexed owner, address indexed spender, uint256 value);                              
  25 }                                                                                                             
  26                                                                                                               
  27 contract MultiSend {                                                                                          
  28   function multiSend(address _token, address[] addresses, uint amount) public {                               
  29     ERC20 token = ERC20(_token);                                                                              
  30     for(uint i =[38;2;248;248;242;48;2;39;40


claude output: 

claude_outputs/post_claude_results_validation_Nested_Call.pkl
key


I need to analyze this contract for Nested Call vulnerabilities, which occur when a contract makes external calls that could lead to unexpected or malicious behavior.

In the MultiSend contract, there are two main functions to examine:

1. `multiSend` function:
   - Makes external calls to an ERC20 token contract via `transferFrom`
   - These calls are inside a loop iterating through an array of addresses

2. `multiSendEth` function:
   - Makes external calls to transfer ETH to multiple addresses
   - These calls are inside a loop
   - Makes a final external call to return remaining ETH to the sender

The key vulnerability here is related to making external calls inside loops. This pattern can lead to:
- Gas limit problems if the array is very large
- Potential for reentrancy if any of the called addresses contain malicious code
- Possible DoS attacks if any transfer fails

Looking at the code:
- In `multiSendEth`, the `addresses[i].transfer()` call allows sending ETH to arbitrary addresses which could be contracts with fallback functions
- The loops themselves with unbounded array lengths create the context for the nested call vulnerability

VULNERABLE_LINES: [addresses[i].transfer(msg.value / addresses.length);, for(uint i = 0; i < addresses.length; i++) {]

In [44]:
import pandas as pd

def concat_final_line_numbers(series):
    """
    Given a Series of string representations of lists or sets,
    evaluate each and combine all items into a set.
    """
    combined = set()
    for item in series:
        if isinstance(item, str):
            try:
                # Evaluate the string to convert it into a Python object.
                evaluated = eval(item)
                # If evaluated is iterable (but not a string), update the set.
                if hasattr(evaluated, '__iter__') and not isinstance(evaluated, str):
                    combined.update(evaluated)
                else:
                    combined.add(evaluated)
            except Exception as e:
                print(f"Error evaluating {item}: {e}")
        elif pd.notna(item):
            try:
                combined.update(item)
            except Exception:
                combined.add(item)
    return str(combined)

# Read the CSV file (update the path if needed)
df = pd.read_csv('vulnerable_verified_smart_contracts_dataset_with_line_numbers_per_vulnerability_with_source_code.csv')

# Group by the compound key and aggregate the columns.
# For 'vulns' and 'type', join unique values separated by a comma.
grouped = df.groupby(['file_path', 'contract_address'], as_index=False).agg({
    'final_line_numbers': concat_final_line_numbers,
    'vuln': lambda x: ','.join(x.dropna().unique()),
    'type': lambda x: ','.join(x.dropna().unique())
})

grouped['source_code'] = None
for index, row in grouped.iterrows():
    source_code_df = df[(df['file_path'] == row['file_path']) & (df['contract_address'] == row['contract_address'])]
    source_code = None
    check_okay = True
    for index_1, row_1 in source_code_df.iterrows():
        if source_code is None:
            source_code = row_1['source_code']
        else:
            if source_code != row_1['source_code']:
                print('something wrong')
                check_okay = False
    if check_okay:
        grouped.loc[index, 'source_code'] = source_code
        
grouped.to_csv('vulnerable_verified_smart_contracts_dataset_with_line_numbers_with_source_code.csv', index=False)

In [46]:
grouped

,file_path,contract_address,final_line_numbers,vuln,type,source_code
0,@c-layer/common/contracts/core/Core.sol,0x2a903c2f657803a2e614c42672247366d757ab34,{55},DC,train,pragma solidity ^0.6.0;\r\n\r\n\r\n\r\n\r\n\r\...
1,A004.sol,0xcae22909c9dbc37c2f6c2780d1f58decd5d3b1fd,"{41, 46}","IOU,NC",train,pragma solidity ^0.4.25;\r\n\r\n\r\n\r\n/**\r\...
2,ABIO_preICO.sol,0x6d84769b1e287a27f282a938c8110b22714dbf78,{201},TD,train,pragma solidity ^0.4.24;\r\ncontract Ownable{\...
3,ACCURAL_DEPOSIT.sol,0x4320e6f8c05b27ab4707cd1f6d5ce6f3e4b3a5a1,"{49, 47}",RE,train,pragma solidity ^0.4.19;\r\n\r\ncontract ACCUR...
4,ACL.sol,0x96f041b96708813b1d789606926c524e78543664,{252},DC,train,//File: contracts/acl/IACL.sol\r\npragma solid...
...,...,...,...,...,...,...
604,ultra_bank.sol,0x094a5e6e395212bfc6c773b2210409eba4ca19d7,"{24, 20, 22}","RE,TD",train,pragma solidity ^0.4.25;\r\n\r\ncontract ultra...
605,we_play.sol,0x6df766e4b524aa1d4b4b9405994ca69dc161da3f,{34},TOD,test,pragma solidity ^0.4.25;\r\n\r\ncontract we_pl...
606,xcat.sol,0xb8aa8971e9201d183d1dadf5acc5c3f6b3076bc0,"{32, 25, 41, 30, 31}","IOU,UpS,TD",train,pragma solidity ^0.4.18;\r\n\r\ncontract HTLC ...
607,xyphar.sol,0x12359487aa7844fead6a20aca5bfae1d3196cedb,{46},NC,train,pragma solidity 0.4.24;\r\n\r\n\r\nlibrary Saf...
